In [ ]:
!pip install av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 61.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from tqdm import tqdm
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# UCF101 returns clips as (T, H, W, C) tensors
transform = transforms.Compose([
    transforms.Lambda(lambda x: x.permute(0, 3, 1, 2)), # Convert to (T, C, H, W)
    transforms.Lambda(lambda x: x / 255.0),            # Normalize to [0, 1]
    transforms.Resize((56, 56)),                     # Resize frames
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

video_path = "/content/drive/MyDrive/ComputerVision/UCF101horse/videos"
annotation_path = "/content/drive/MyDrive/ComputerVision/UCF101horse/annotations"

ucf101_dataset = datasets.UCF101(
    root=video_path,
    annotation_path=annotation_path,
    frames_per_clip=32,
    step_between_clips=32, # no overlap
    fold=1, # 1, 2, or 3
    train=True,
    transform=transform
)

dataloader = DataLoader(ucf101_dataset, batch_size=8, shuffle=True)

print(f"Dataset size: {len(ucf101_dataset)}")

100%|██████████| 18/18 [00:08<00:00,  2.07it/s]

Dataset size: 1394


In [ ]:
for video, _, label in dataloader:
  print(video.shape)
  print(label)
  break

torch.Size([8, 32, 3, 56, 56])
tensor([0, 0, 1, 1, 1, 1, 1, 0])


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv3d(3, 6, 5)
        self.conv2 = nn.Conv3d(6, 16, 5)
        self.pool = nn.MaxPool3d(2)
        self.fc1 = nn.Linear(9680, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

vc = Net().cuda()

optimizer = torch.optim.Adam(vc.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

for i in range(100):
  vc.train()
  loss_sum = 0
  correct_sum = 0
  total_sum = 0
  for x, _, y in tqdm(dataloader):
    optimizer.zero_grad()
    outputs = vc(x.permute(0,2,1,3,4).cuda())
    loss = criterion(outputs, y.cuda())
    loss.backward()
    optimizer.step()

    y_ = torch.argmax(outputs, dim=1)
    correct_sum += torch.sum((y_ == y.cuda()).float()).item()
    total_sum += len(x)
    loss_sum += len(x)*loss.item()


  print(loss_sum/total_sum, correct_sum/total_sum)

100%|██████████| 175/175 [00:53<00:00,  3.25it/s]


0.5027634134743105 0.7431850789096126


100%|██████████| 175/175 [00:53<00:00,  3.29it/s]


0.5591464104372881 0.6527977044476327


100%|██████████| 175/175 [00:53<00:00,  3.28it/s]


...

0.0005976655922501965 1.0


100%|██████████| 175/175 [00:53<00:00,  3.28it/s]


0.0003573451769614505 1.0


 93%|█████████▎| 163/175 [00:51<00:03,  3.19it/s]


KeyboardInterrupt: 